# Titanic — exploratory data analysis

Uses Seaborn's bundled `titanic` dataset (`sns.load_dataset("titanic")`). Target: **survived** (0/1).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 5)

df = sns.load_dataset("titanic")
df.head()

## Shape, dtypes, missing values

In [ ]:
print(df.shape)
display(df.dtypes.to_frame("dtype"))
missing = df.isna().mean().sort_values(ascending=False)
missing.plot(kind="bar", title="Fraction missing per column", color="steelblue")
plt.ylabel("fraction")
plt.tight_layout()
plt.show()

In [ ]:
df.describe(include="all").T

## Class balance

In [ ]:
rate = df["survived"].mean()
fig, ax = plt.subplots()
df["survived"].value_counts().sort_index().plot(kind="bar", ax=ax, color=["#c44e52", "#4c72b0"])
ax.set_xticklabels(["did not survive", "survived"], rotation=0)
ax.set_title(f"Survival counts (overall survival rate ≈ {rate:.2%})")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()

## Sex and passenger class

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.countplot(data=df, x="sex", hue="survived", ax=axes[0], palette="Set2")
axes[0].set_title("Survival by sex")
sns.countplot(data=df, x="pclass", hue="survived", ax=axes[1], palette="Set2")
axes[1].set_title("Survival by passenger class")
plt.tight_layout()
plt.show()

## Age distribution by outcome

In [ ]:
fig, ax = plt.subplots()
sns.histplot(data=df, x="age", hue="survived", kde=True, bins=30, ax=ax, palette="muted")
ax.set_title("Age distribution by survival")
plt.tight_layout()
plt.show()

## Fare

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(data=df, x="survived", y="fare", ax=axes[0], palette="pastel")
axes[0].set_title("Fare vs survival (boxplot)")
sns.violinplot(data=df, x="pclass", y="fare", hue="survived", split=True, ax=axes[1], palette="muted")
axes[1].set_title("Fare by class and survival")
plt.tight_layout()
plt.show()

## Embarkation port

In [ ]:
tmp = df.dropna(subset=["embarked"])
fig, ax = plt.subplots()
sns.barplot(data=tmp, x="embarked", y="survived", ax=ax, errorbar=("ci", 95), palette="rocket")
ax.set_title("Mean survival rate by embarked (95% CI)")
ax.set_ylabel("survival rate")
plt.tight_layout()
plt.show()

## Numeric correlations

In [ ]:
num_cols = ["survived", "pclass", "age", "sibsp", "parch", "fare"]
corr = df[num_cols].corr()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Correlation matrix (numeric features)")
plt.tight_layout()
plt.show()

## Pairplot (sample)

Downsample for speed if needed.

In [ ]:
sample = df.dropna(subset=["age", "fare"]).sample(n=min(400, len(df)), random_state=42)
sns.pairplot(sample, hue="survived", vars=["age", "fare", "pclass"], corner=True, palette="Set2")
plt.suptitle("Pairplot (subset)", y=1.02)
plt.show()

## Takeaways

- **Sex** and **pclass** are strong separators in this dataset.
- **Age** and **fare** show overlap but different shapes by outcome.
- **deck** / **age** have notable missingness; training scripts use an explicit feature subset and imputation — see `train.py`.